# CLB
## Dependencies

In [15]:
import os
import tqdm
import shutil
import numpy as np
import scipy.io
import pandas as pd
import pickle
import h5py

import logging

In [16]:
def load_mocap_data(file_path):
    mat = scipy.io.loadmat(file_path)
    exp_name = [m for m in mat.keys() if m[:2] != '__'][0]  ## TOCHANGE
    data = np.moveaxis(mat[exp_name][0, 0]['Trajectories'][0, 0]['Labeled']['Data'][0, 0], 2, 0)
    keypoints = [label[0].replace('coordinate', 'coord') for label in
                    mat[exp_name][0, 0]['Trajectories'][0, 0]['Labeled']['Labels'][0, 0][0]]

    # Very important. Make sure the keypoints are always in the same order even if not saved so in the original files
    new_order = np.argsort(keypoints)
    keypoints = [keypoints[n] for n in new_order]
    data = data[:, new_order, :]

    return data, keypoints

## Load data

In [17]:
# check file names
file_path = "/home/group-cvg/cvg-rose/bogna_mocap/CP1B_CLB/"
files = os.listdir(file_path)
files.remove("CP1B_S1_M1_MC4_CLB_31_01_2019_12_00_proc_bij_2_04_2019_D.mat")
files

['CP1B_S17_M5_MC5_CLB_7_02_2019_1440_proc_bij_14_04_2019_B.mat',
 'CP1B_S11_M5_MC5_CLB_4_02_2019_14_15_proc_bij_5_4_2019_C.mat',
 'CP1B_S14_M2_MC4_CLB_7_02_2019_11_30_proc_bij_9_4_2019_C.mat',
 'CP1B_S8_M2_MC4_CLB_4_02_2019_11_00_proc_bij_5_4_2019_D.mat',
 'CP1B_S10_M4_MC5_CLB_4_02_2019_13_15_proc_BIJ_5.04_2019_B.mat',
 'CP1B_S19_M1_MC4_CLB_12_02_2019_10_00_proc_bij_30_05_2019_A.mat',
 'CP1B_S2_M2_MC4_CLB_31_01_2019_14_00_proc_bij_3_4_2019_A.mat',
 'CP1B_S16_M4_MC5_CLB_7_02_2019_1400_proc_bij_12_4_2019_A.mat',
 'CP1B_S6_M6_MC5_CLB_1_02_2019_17_00_proc_bij_4_4_2019_A.mat',
 'CP1B_S4_M4_MC5_CLB_1_02_2019_15_00_proc_bij_3_4_2019_C.mat',
 'CP1B_S15_M3_MC4_CLB_7_02_2019_12_30_proc_bij_11_4_2019_D.mat',
 'CP1B_S25_M1_MC4_CLB_13_02_2019_13_00_proc_bij_31_05_2019_D_replication_noCOORD.mat',
 'CP1B_S21_M3_MC4_CLB_12_02_2019_14_00_proc_bij_31_05_2019_C.mat',
 'CP1B_S12_M6_MC5_CLB_4_02_2019_15_00_proc_bij_5_4_2019_D.mat',
 'CP1B_S3_M3_MC4_CLB_31_01_2019_16_00_proc_bij_3_4_2019_B.mat',
 'CP1B_S26_

In [37]:
data_CP1B_CLB = {}

for file_name in files:
    file = os.path.join(file_path, file_name)
    
    if os.path.isfile(file):
        data, keypoints = load_mocap_data(file)
        key_name = str(file_name.split("_")[0] + "_" + file_name.split("_")[1]) + "_CLB"
        data_CP1B_CLB[key_name] ={"label": None, "keypoints": None}
        data_CP1B_CLB[key_name]["label"] = file_name.split(".")[-2][-1]
        data_CP1B_CLB[key_name]["keypoints"] = keypoints
        # 1. Discard last dimension
        data = data[ : , : , :3]
        
        # 2. Remove first 1500 frames from each sequence
        data = data[1500:]
        
        data_CP1B_CLB[key_name]["data"] = data

In [38]:
data_CP1B_CLB['CP1B_S13_CLB']["label"] = "B"
data_CP1B_CLB['CP1B_S25_CLB']["label"] = "D"
data_CP1B_CLB['CP1B_S26_CLB']["label"] = "B"

In [ ]:
# The correct keypoint 
keypoints_name = ['left_ankle',  'left_back',  'left_coord',  'left_hip',  'left_knee', 
                  'right_ankle', 'right_back', 'right_coord', 'right_hip', 'right_knee']

for key, value in data_CP1B_CLB.items():
    
    # Create an empty array to store
    arr = np.full((34500, 10, 3), np.nan)
    i = 0  # index of correct keypoint
    j = 0  # index of actual keypoint
    while i < 10:
        if keypoints_name[i] == value["keypoints"][i]:
            arr[:, i, :] = value["data"][:, j, :]
            i += 1 
            j += 1
        else:
            value["keypoints"].insert(i, None)
            i += 1
    # remove frames with too much missing
    missing_per_sample = np.isnan(arr).sum(axis=(1, 2))
    indices = np.where(missing_per_sample<=20)
    value["data"] = arr[indices]
    l = len(value["data"])
    l = (l // 10) * 10
    value["data"] = value["data"][:l]

In [42]:
import pickle

with open('../../data/CP1B_CLB.pkl', 'wb') as file:
    pickle.dump(data_CP1B_CLB, file)

## Some further process

In [1]:
import pickle

# Read data
with open('/home/rguo_hpc/myfolder/code/mocap/data/CP1B_CLB.pkl', 'rb') as file:
    data_CP1B_CLB = pickle.load(file)

In [ ]:
for key, value in data_CP1B_CLB.items():
    print(key, value["label"], value["keypoints"], value["data"].shape)

In [2]:
len(data_CP1B_CLB.keys())

48

In [ ]:
original_keys = list(data_CP1B_CLB.keys())
for seq_name in original_keys:
    value = data_CP1B_CLB[seq_name]
    data_CP1B_CLB[seq_name + "_1"] = {"label": value["label"], "data": value["data"][:18000],  "keypoints": value["keypoints"]}
    data_CP1B_CLB[seq_name + "_2"] = {"label": value["label"], "data": value["data"][-18000:], "keypoints": value["keypoints"]}
    del data_CP1B_CLB[seq_name]

# FL2